In [1]:
import pandas as pd
pairs = pd.read_csv("./datas/comovement_pairs.csv")

In [2]:
from modules.get_data_table import get_base_data
data = get_base_data()

In [3]:
from modules.get_data_table import VALUE_PIVOUT, QUANTITY_WEIGHT_PIVOUT
value_pivot = VALUE_PIVOUT(data)
weight_quantity_pivot= QUANTITY_WEIGHT_PIVOUT(data)

value_pivot.head()

,ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,hs4,,,,,,,,,,,,,,,,,,,,,
AANGBULD,4810,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,2102,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,4403,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,8105,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,2814,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0


In [ ]:
X_train, y_train = pd.read_csv("./datas/X_train.csv"), pd.read_csv("./datas/y_train.csv")
X_valid, y_valid = pd.read_csv("./datas/X_valid.csv"), pd.read_csv("./datas/y_valid.csv")
X_test, y_test = pd.read_csv("./datas/X_test.csv"), pd.read_csv("./datas/y_test.csv")

In [ ]:
from modules.scaler import get_scaler

scaler = get_scaler(X_train.values)
scaled_X_train = scaler.transform(X_train.values)
scaled_X_valid = scaler.transform(X_valid.values)
scaled_X_test = scaler.transform(X_test.values)

In [6]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
        random_state=42,
        force_col_wise=True
    )
model.fit(pd.DataFrame(scaled_X_train, columns=X_train.columns), y_train.values.ravel())
pred = model.predict(pd.DataFrame(scaled_X_valid, columns=X_valid.columns))

[LightGBM] [Info] Total Bins 3243
[LightGBM] [Info] Number of data points in the train set: 29125, number of used features: 15
[LightGBM] [Info] Start training from score 3740061.937270


In [7]:
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error, PredictionErrorDisplay
import numpy as np

mse = mean_squared_error(y_valid, np.array(pred))
r2 = r2_score(y_valid, np.array(pred))
rmse = root_mean_squared_error(y_valid, np.array(pred))
print(f"MSE: {mse}, RMSE: {rmse}, R2: {r2}")

MSE: 1139265676699.9714, RMSE: 1067363.8914165925, R2: 0.9917217628436684


In [8]:
import importlib
import modules.create_submit as cs
importlib.reload(cs)

submission = cs.create_submit(value_pivot, weight_quantity_pivot, pairs, data, model, scaler)
print(f"\n✅ 예측 완료! 제출 데이터 크기: {len(submission)}개")
submission.head()


✅ 예측 완료! 제출 데이터 크기: 1309개


,leading_item_id,following_item_id,value
0,AANGBULD,DDEXPPXU,30717
1,AANGBULD,GKQIJYDH,10076701
2,AANGBULD,GYHKIVQT,27779381
3,AANGBULD,HCDTGMST,77508
4,AANGBULD,IGDVVKUD,36949


In [9]:
# import score
# import importlib
# importlib.reload(score)
# from score import print_validation_summary

# y_test = data.loc[data['ym'] == pd.to_datetime('2025-07-01')]
# print_validation_summary(y_test, submission, pairs)

In [9]:
submission.to_csv('./submit-4.csv', index=False)

In [11]:
submission.shape

(1444, 3)